# CUDA environment and Deterministic

In [ ]:
# CUDA environment
# Deterministic

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import gc
import torch
from transformers import logging as tf_logging
import numpy as np
import random
import logging

# off warming
tf_logging.set_verbosity_error()
logging.disable(logging.WARNING)

# Deterministic
seed = 42
random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

from tqdm.autonotebook import tqdm
import pandas as pd
from datasets import Dataset
import transformers
import dspy
from sklearn import metrics
import datetime

# Load dataset

In [ ]:

df_train = pd.read_csv('./dataset/radnlp_2024_train_val_test/en/main_task/train/label.csv', encoding='utf-8')
for i, idx in enumerate(df_train['id'].values):
    with open(f'./dataset/radnlp_2024_train_val_test/en/main_task/train/{idx}.txt') as file:
        text = "".join(file.readlines()).replace("\n", " ").replace("  ", " ").strip()
    df_train.loc[i, 'report'] = text


df_val_as_test = pd.read_csv('./dataset/radnlp_2024_train_val_test/en/main_task/val/label.csv', encoding='utf-8')
for i, idx in enumerate(df_val_as_test['id'].values):
    with open(f'./dataset/radnlp_2024_train_val_test/en/main_task/val/{idx}.txt') as file:
        text = "".join(file.readlines())#.replace("\n", " ").replace("  ", " ").strip()
    df_val_as_test.loc[i, 'report'] = text


df_test = pd.read_csv('./dataset/radnlp_2024_train_val_test/en/main_task/test/sample_submission.csv', encoding='utf-8')
for i, idx in enumerate(df_test['id'].values):
    with open(f'./dataset/radnlp_2024_train_val_test/en/main_task/test/{idx}.txt') as file:
        text = "".join(file.readlines())#.replace("\n", " ").replace("  ", " ").strip()
    df_test.loc[i, 'report'] = text
    

In [ ]:
df_train.info()

In [ ]:
df_val_as_test.info()

In [ ]:
df_test.info()

# Make Dataloader

In [ ]:
df_val = df_train.sample(frac=0.5, random_state=seed)
print(f'{len(df_val)=}')
df_train_par = df_train.drop(index=df_val.index)
print(f'{len(df_train_par)=}')

In [ ]:
from dspy.datasets import DataLoader

dl = DataLoader()

trainset = dl.from_pandas(
    df_train_par,
    fields=(['report', 't', 'n', 'm']),
    input_keys=(['report'])
)

valset = dl.from_pandas(
    df_val,
    fields=(['report', 't', 'n', 'm']),
    input_keys=(['report'])
)

testset = dl.from_pandas(
    df_val_as_test,
    fields=(['report']),
    input_keys=(['report'])
)

testset_sub = dl.from_pandas(
    df_test,
    fields=(['report']),
    input_keys=(['report'])
)


In [ ]:
print(f'=== {trainset[0]=}\n\n=== {valset[0]=}\n\n=== {testset[0]=}')

# DSPy Model Config

In [ ]:
import litellm
litellm.set_verbose=False
litellm.suppress_debug_info = True

from dotenv import load_dotenv

# 加載 .env 檔案
load_dotenv()

gpt4o = {'model_name':'openai/gpt-4o', 'url':'', 'api_key':os.getenv("OPENAI_API_KEY")}
gemini2 = {'model_name':'gemini/gemini-2.0-flash-exp', 'url':'', 'api_key':os.getenv("GEMINI_API_KEY")}

lm_gpt4o = dspy.LM(f"{gpt4o['model_name']}", api_key=gpt4o['api_key'], cache=True)
lm_gemini2 = dspy.LM(f"{gemini2['model_name']}", api_key=gemini2['api_key'], cache=True)

dspy.configure(lm=lm_gpt4o,)



# Prompt Design

In [ ]:
from typing import TypedDict, List, Literal
from pydantic import BaseModel


class DetermineStaging(dspy.Signature):
    """You are an senior radiologist and oncologist. You are always carefully read the entire radiology report to identify any mentions of tumor size, location, lymph node involvement, and distant metastasis. 
    The information not disclosed in the report is that there is no evidence of tumor size, location, lymph node involvement, and distant metastasis. 
    The Goal of task: Determine the T, N, and M staging of lung cancer from the radiology report according to the IASLC 8th edition lung cancer staging system.
    The Prerequisite: You only have the radiology text report, no X-ray/CT image files. No further X-ray or CT scan will be processing.
    When staging tumors, prioritize Tis classification for non-invasive cases regardless of size, even if dimensions approach T1mi, T1a, T1b, T1c criteria. Definitive pathological evidence of invasion is required for T1mi, T1a, T1b, T1c or higher classifications, and any uncertainty should default to the more conservative staging option.
    
    Hints from the IASLC 8th edition lung cancer staging system:
    
    Primary Tumor (T) Staging: In cases of uncertainty regarding invasion or measurements, choose the most conservative stage. When describing tumor dimensions in the report, always distinguish between total size and solid component size, and use the solid component for staging where applicable.
    - T0: no evidence of a primary tumor
    - Tis: carcinoma in situ or no invasive component at histopathology. limited to the epithelium, no evidence of invasion into the underlying tissue or stroma. Tumors without invasive features must be classified as Tis.
    - T1mi: minimally invasive adenocarcinoma, tumor has an invasive component measuring <=5 mm at histopathology. Requires pathological confirmation of invasion; otherwise, classify as Tis.
    - T1a: tumor <=10 mm in greatest dimension. no deeper invasion.
    - T1b: tumor <=20 mm and >10 mm but in greatest dimension. with evidence of invasion.
    - T1c: tumor <=30 mm and >20 mm but in greatest dimension
    - T2a: tumor <=40 mm and >30 mm in greatest dimension, tumor involving: visceral pleura, main bronchus (not carina), atelectasis to hilum
    - T2b: tumor <=50 mm and >40 mm in greatest dimension, tumor involving: visceral pleura, main bronchus (not carina), atelectasis to hilum
    - T3: tumor <=70 mm and >50 mm in greatest dimension or invading chest wall, pericardium, phrenic nerve; or separate tumor nodule(s) in the same lobe
    - T4: tumor >70 mm in greatest dimension or invading mediastinum, diaphragm, heart, great vessels, recurrent laryngeal nerve, trachea, esophagus, spine; or tumor nodule(s) in a different ipsilateral lobe.
    
    Lymph Node Metastasis (N) Staging: 
    - N0: No regional lymph node metastasis. Use only when lymph nodes are clearly normal without any suspicion of metastasis. 
    - N1: Ipsilateral bronchial and hilar lymph node metastasis. (ipsilateral peribronchial, ipsilateral hilar). Classify as N1 if there is mention of 'prominent hilar lymph nodes' or 'suspicion of metastasis in the hilar nodes'. 
    - N2: Ipsilateral mediastinal and subcarinal lymph node metastasis. (ipsilateral mediastinal, subcarina). Classify as N2 if there is mention of 'enlarged mediastinal lymph nodes' or 'cannot rule out metastasis in mediastinal lymph nodes.' 
    - N3: Contralateral mediastinal and hilar, scalene or supraclavicular lymph node metastasis. (contralateral mediastinal or hilar, scalene or supraclavicular) 
    
    Distant Metastasis (M) Staging: 
    - M0: No distant metastasis. Use this classification only when there is no evidence of distant metastasis, including pleural dissemination, malignant pleural effusion, or any organ involvement. 
    - M1a: Pleural dissemination (malignant pleural effusion, pericardial effusion, or pleural nodules), or separate tumor nodules in a contralateral lobe. Include cases with pleural effusion confirmed as malignant or pleural-based nodules. - Also include separate nodules identified in the contralateral lung. - If additional evidence suggests metastases in other organs, classify as M1c instead. 
    - M1b: Single organ distant metastasis (outside lung/pleura). Classify as M1b if a single organ (e.g., liver, bone, brain, or adrenal gland) shows definitive evidence of metastasis. Ensure that no metastases are present in other organs. If multiple organs are involved, reclassify as M1c. 
    - M1c: Multiple organ distant metastasis. Use this classification when metastases are identified in two or more organs (e.g., bone, liver, adrenal glands). - Include cases with pleural dissemination if other organs are also involved. Examples include bilateral adrenal gland metastases or widespread bone lesions. 
    """
    
    report: str = dspy.InputField(desc="the radiology report")
    
    T_staging: Literal['T0', 'Tis', 'T1mi', 'T1a', 'T1b', 'T1c', 'T2a', 'T2b', 'T3', 'T4'] = dspy.OutputField(desc="""Primary Tumor (T) Staging: In cases of uncertainty regarding invasion or measurements, choose the most conservative stage. When describing tumor dimensions in the report, always distinguish between total size and solid component size, and use the solid component for staging where applicable.
    - T0: no evidence of a primary tumor
    - Tis: carcinoma in situ or no invasive component at histopathology. limited to the epithelium, no evidence of invasion into the underlying tissue or stroma. Tumors without invasive features must be classified as Tis.
    - T1mi: minimally invasive adenocarcinoma, tumor has an invasive component measuring <=5 mm at histopathology. Requires pathological confirmation of invasion; otherwise, classify as Tis.
    - T1a: tumor <=10 mm in greatest dimension. no deeper invasion.
    - T1b: tumor <=20 mm and >10 mm but in greatest dimension. with evidence of invasion.
    - T1c: tumor <=30 mm and >20 mm but in greatest dimension
    - T2a: tumor <=40 mm and >30 mm in greatest dimension, tumor involving: visceral pleura, main bronchus (not carina), atelectasis to hilum
    - T2b: tumor <=50 mm and >40 mm in greatest dimension, tumor involving: visceral pleura, main bronchus (not carina), atelectasis to hilum
    - T3: tumor <=70 mm and >50 mm in greatest dimension or invading chest wall, pericardium, phrenic nerve; or separate tumor nodule(s) in the same lobe
    - T4: tumor >70 mm in greatest dimension or invading mediastinum, diaphragm, heart, great vessels, recurrent laryngeal nerve, trachea, esophagus, spine; or tumor nodule(s) in a different ipsilateral lobe.
    """)
    
    N_staging: Literal['N0', 'N1', 'N2', 'N3'] = dspy.OutputField(desc="""Lymph Node Metastasis (N) Staging: 
    - N0: No regional lymph node metastasis. Use only when lymph nodes are clearly normal without any suspicion of metastasis. 
    - N1: Ipsilateral bronchial and hilar lymph node metastasis. (ipsilateral peribronchial, ipsilateral hilar). Classify as N1 if there is mention of 'prominent hilar lymph nodes' or 'suspicion of metastasis in the hilar nodes'. 
    - N2: Ipsilateral mediastinal and subcarinal lymph node metastasis. (ipsilateral mediastinal, subcarina). Classify as N2 if there is mention of 'enlarged mediastinal lymph nodes' or 'cannot rule out metastasis in mediastinal lymph nodes.' 
    - N3: Contralateral mediastinal and hilar, scalene or supraclavicular lymph node metastasis. (contralateral mediastinal or hilar, scalene or supraclavicular)
    """)
    
    M_staging: Literal['M0', 'M1a', 'M1b', 'M1c'] = dspy.OutputField(desc="""Distant Metastasis (M) Staging: 
    - M0: No distant metastasis. Use this classification only when there is no evidence of distant metastasis, including pleural dissemination, malignant pleural effusion, or any organ involvement. 
    - M1a: Pleural dissemination (malignant pleural effusion, pericardial effusion, or pleural nodules), or separate tumor nodules in a contralateral lobe. Include cases with pleural effusion confirmed as malignant or pleural-based nodules. - Also include separate nodules identified in the contralateral lung. - If additional evidence suggests metastases in other organs, classify as M1c instead. 
    - M1b: Single organ distant metastasis (outside lung/pleura). Classify as M1b if a single organ (e.g., liver, bone, brain, or adrenal gland) shows definitive evidence of metastasis. Ensure that no metastases are present in other organs. If multiple organs are involved, reclassify as M1c. 
    - M1c: Multiple organ distant metastasis. Use this classification when metastases are identified in two or more organs (e.g., bone, liver, adrenal glands). - Include cases with pleural dissemination if other organs are also involved. Examples include bilateral adrenal gland metastases or widespread bone lesions.
    """)

class DetermineStagingConclude(DetermineStaging): 
    """The intern doctor determined the stage of lung cancer from radiology reports. 
    I can critically assess the provided intern doctor opinions.
    """ 

    context = dspy.InputField(desc="A list of opinions from intern doctors.") 
    
    T_staging: Literal['T0', 'Tis', 'T1mi', 'T1a', 'T1b', 'T1c', 'T2a', 'T2b', 'T3', 'T4'] = dspy.OutputField(desc="""Primary Tumor (T) Staging: In cases of uncertainty regarding invasion or measurements, choose the most conservative stage. When describing tumor dimensions in the report, always distinguish between total size and solid component size, and use the solid component for staging where applicable.
    - T0: no evidence of a primary tumor
    - Tis: carcinoma in situ or no invasive component at histopathology. limited to the epithelium, no evidence of invasion into the underlying tissue or stroma. Tumors without invasive features must be classified as Tis.
    - T1mi: minimally invasive adenocarcinoma, tumor has an invasive component measuring <=5 mm at histopathology. Requires pathological confirmation of invasion; otherwise, classify as Tis.
    - T1a: tumor <=10 mm in greatest dimension. no deeper invasion.
    - T1b: tumor <=20 mm and >10 mm but in greatest dimension. with evidence of invasion.
    - T1c: tumor <=30 mm and >20 mm but in greatest dimension
    - T2a: tumor <=40 mm and >30 mm in greatest dimension, tumor involving: visceral pleura, main bronchus (not carina), atelectasis to hilum
    - T2b: tumor <=50 mm and >40 mm in greatest dimension, tumor involving: visceral pleura, main bronchus (not carina), atelectasis to hilum
    - T3: tumor <=70 mm and >50 mm in greatest dimension or invading chest wall, pericardium, phrenic nerve; or separate tumor nodule(s) in the same lobe
    - T4: tumor >70 mm in greatest dimension or invading mediastinum, diaphragm, heart, great vessels, recurrent laryngeal nerve, trachea, esophagus, spine; or tumor nodule(s) in a different ipsilateral lobe.
    """)
    
    N_staging: Literal['N0', 'N1', 'N2', 'N3'] = dspy.OutputField(desc="""Lymph Node Metastasis (N) Staging: 
    - N0: No regional lymph node metastasis. Use only when lymph nodes are clearly normal without any suspicion of metastasis. 
    - N1: Ipsilateral bronchial and hilar lymph node metastasis. (ipsilateral peribronchial, ipsilateral hilar). Classify as N1 if there is mention of 'prominent hilar lymph nodes' or 'suspicion of metastasis in the hilar nodes'. 
    - N2: Ipsilateral mediastinal and subcarinal lymph node metastasis. (ipsilateral mediastinal, subcarina). Classify as N2 if there is mention of 'enlarged mediastinal lymph nodes' or 'cannot rule out metastasis in mediastinal lymph nodes.' 
    - N3: Contralateral mediastinal and hilar, scalene or supraclavicular lymph node metastasis. (contralateral mediastinal or hilar, scalene or supraclavicular)
    """)
    
    M_staging: Literal['M0', 'M1a', 'M1b', 'M1c'] = dspy.OutputField(desc="""Distant Metastasis (M) Staging: 
    - M0: No distant metastasis. Use this classification only when there is no evidence of distant metastasis, including pleural dissemination, malignant pleural effusion, or any organ involvement. 
    - M1a: Pleural dissemination (malignant pleural effusion, pericardial effusion, or pleural nodules), or separate tumor nodules in a contralateral lobe. Include cases with pleural effusion confirmed as malignant or pleural-based nodules. - Also include separate nodules identified in the contralateral lung. - If additional evidence suggests metastases in other organs, classify as M1c instead. 
    - M1b: Single organ distant metastasis (outside lung/pleura). Classify as M1b if a single organ (e.g., liver, bone, brain, or adrenal gland) shows definitive evidence of metastasis. Ensure that no metastases are present in other organs. If multiple organs are involved, reclassify as M1c. 
    - M1c: Multiple organ distant metastasis. Use this classification when metastases are identified in two or more organs (e.g., bone, liver, adrenal glands). - Include cases with pleural dissemination if other organs are also involved. Examples include bilateral adrenal gland metastases or widespread bone lesions.
    """)

class DetermineCOTandConclude(dspy.Module):
    def __init__(self, **kwargs): 
        self.determine = [dspy.ChainOfThought(DetermineStaging, **kwargs) for i in range(5)] 
        self.conclude = dspy.ChainOfThought(DetermineStagingConclude)#, **kwargs) 
    
    def forward(self, report): 
        kwargs_report = dict(report=report) 
        opinions = []
        for i, c in enumerate(self.determine):
            if i%2 == 0:
                with dspy.context(lm=lm_gpt4o):
                    opinions.append(c(**kwargs_report))
            else:
                try:
                    with dspy.context(lm=lm_gemini2):
                        opinions.append(c(**kwargs_report))
                except:
                    with dspy.context(lm=lm_gpt4o):
                        opinions.append(c(**kwargs_report))

        opinions = [[opinion.reasoning.replace('\n', ' ').strip('.'), opinion.T_staging, opinion.N_staging, opinion.M_staging] for opinion in opinions] 
        opinions = [f"I am an intern doctor, and the reason for my decision to stage is: {reason}. Therefore, my answer is {T}, {N}, {M}." for reason, T, N, M in opinions] 

        result = self.conclude(context=opinions , **kwargs_report)
        return dspy.Prediction(t=result.T_staging, n=result.N_staging, m=result.M_staging, reasoning=result.reasoning)

        


## Validation function

In [ ]:
def validate_answer_accuracy(expect, predict, trace=None):
    from sklearn.metrics import accuracy_score

    try:
        score = accuracy_score([expect[k] for k in ['t', 'n', 'm']], 
                            [predict[k] for k in ['t', 'n', 'm']], 
                              )
    except:
        #print(f'Error')
        score = 0
    
    return score

## try an example

In [ ]:
%%time

from dspy.evaluate import SemanticF1, answer_exact_match


example = trainset[0]

classifyCONCLUDE = DetermineCOTandConclude(temperature=random.uniform(0.001, 0.85)) 

# Instantiate the metric.
metric = validate_answer_accuracy #SemanticF1()

# Produce a prediction from our `cot` module, using the `example` above as input.
classified = classifyCONCLUDE(report=example.report)

print(f"Report:\t{example.report}\n")
print(f"Gold Reponse:\t{example.t, example.n, example.m}\n")
print(f"Predicted Response:\t{classified.t, classified.n, classified.m}\n")
print(f"Reasoning:\t{classified.reasoning}\n")

# Compute the metric score for the prediction.
score = metric(example, classified)
print(f"F2 Accuracy Score:\t{score}")


# Optimization

In [ ]:
print(f'{len(trainset)=}\n{len(valset)=}\n{len(testset)=}\n{len(testset_sub)=}')

In [ ]:
%%time

import random

max_bootstrapped_demos = 50
max_labeled_demos = 50

metric = validate_answer_accuracy

optimizer = dspy.MIPROv2(metric=metric, auto="medium", num_threads=1,)  # use fewer threads if your rate limit is small

trainset_random = random.sample(trainset, len(trainset))
valset_random = random.sample(valset, len(valset))

optimized_classify = optimizer.compile(classifyCONCLUDE, 
                                      trainset=trainset_random, 
                                      valset=valset_random,
                                      max_bootstrapped_demos=max_bootstrapped_demos, 
                                      max_labeled_demos=max_labeled_demos,
                                      requires_permission_to_run=False, 
                                       seed=seed,
                                      )


In [ ]:
# Define an evaluator that we can re-use.
evaluate = dspy.Evaluate(devset=valset[:10], metric=metric, num_threads=1,
                         display_progress=True, display_table=3)
evaluate(optimized_classify)

# Prediction

In [ ]:
%%time

from tqdm import tqdm
import time


predicts = {'id':[], 't':[], 'n':[], 'm':[]}

for i, example in enumerate(tqdm(testset_sub)):
    idx = df_test.loc[i, 'id']
    predicts['id'].append(idx)
    try:
        predict = optimized_classify(report=example.report,)
        predicts['t'].append(predict.t)
        predicts['n'].append(predict.n)
        predicts['m'].append(predict.m)
    except:
        predicts['t'].append(f'ERROR: {i}')
        predicts['n'].append(f'ERROR: {i}')
        predicts['m'].append(f'ERROR: {i}')
    #time.sleep(30)


## Save Optimized prompt

In [ ]:
log_time = datetime.datetime.now().strftime('%y%m%d.%H%M')
model_name = gpt4o['model_name']

try:
    filename_opt = f'maintask_en_CONCLUDE_{max_bootstrapped_demos}shots_{model_name.replace("/", "-")}_{log_time}.json' # .pkl
    optimized_classify.save(filename_opt)
except:
    filename_opt = f'maintask_en_CONCLUDE_{max_bootstrapped_demos}shots_{model_name.replace("/", "-")}_{log_time}.pkl' # .pkl
    optimized_classify.save(filename_opt)

print(f'{model_name=}\n{log_time=}\n{filename_opt=}')


## Prediction to dataframe

In [ ]:
df_expect = df_val_as_test[['id', 't', 'n', 'm']].copy(deep=True)

df_predict = pd.DataFrame(columns=['id','t','n','m'])
df_predict['id'] = predicts['id']
df_predict['t'] = predicts['t']
df_predict['n'] = predicts['n']
df_predict['m'] = predicts['m']

df_predict_val = df_predict[df_predict['id'].isin(df_val_as_test['id'].values)].copy(deep=True)

# Validate Public (validata) performance

In [ ]:
def MetricRADNLPMainTask(data_expect, data_predict):
    from sklearn.metrics import accuracy_score
    
    scores = {}
    
    scores["Joint accuracy (fine)"] = accuracy_score(["".join(item) for item in data_expect.values.tolist()], 
                                                  ["".join(item) for item in data_predict.values.tolist()], 
                                                    )
    
    data_expect['tc'] = data_expect['t'].map(lambda x: x[:2]).map(lambda x: 'T1' if x=='Ti' else x).values.tolist()
    data_expect['nc'] = data_expect['n'].values.tolist()
    data_expect['mc'] = data_expect['m'].map(lambda x: x[:2]).values.tolist()
    data_predict['tc'] = data_predict['t'].map(lambda x: x[:2]).map(lambda x: 'T1' if x=='Ti' else x).values.tolist()
    data_predict['nc'] = data_predict['n'].values.tolist()
    data_predict['mc'] = data_predict['m'].map(lambda x: x[:2]).values.tolist()
    
    for column in ['t','n','m']:
        expect = data_expect[[column]].values.tolist()
        predict = data_predict[[column]].values.tolist()       
        scores[f'{column.upper()} accuracy (fine)'] = accuracy_score(expect, predict)
        
    scores["Joint accuracy (coarse)"] = accuracy_score(["".join(item) for item in data_expect[['tc','nc','mc']].values.tolist()], 
                                                  ["".join(item) for item in data_predict[['tc','nc','mc']].values.tolist()], 
                                                    )
    
    for column in ['tc','nc','mc']:
        expect = data_expect[[column]].values.tolist()
        predict = data_predict[[column]].values.tolist()       
        scores[f'{column[0].upper()} accuracy (coarse)'] = accuracy_score(expect, predict)
    
    return scores


In [ ]:
columns = ['t','n','m']
print(f'=== {lm_gpt4o.history[-1]["model"]} , {max_labeled_demos}===')
MetricRADNLPMainTask(df_expect[columns], df_predict_val[columns])

## Error vis. and check

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

for column in ['t','n','m']:
    print(f'=== {column} ===')
    expect = df_expect[column].values.tolist()
    predict = df_predict_val[column].values.tolist() 
    target_names = {'t': ['T0', 'Tis', 'T1mi', 'T1a', 'T1b', 'T1c', 'T2a', 'T2b', 'T3', 'T4'],
                    'n': ['N0', 'N1', 'N2', 'N3'] ,
                    'm': ['M0', 'M1a', 'M1b', 'M1c']}
    target_names = df_expect[column].unique().tolist()
    target_names.extend(df_predict_val[column].unique().tolist())
    target_names = list(set(target_names))
    print(classification_report(expect, predict, zero_division=0.0))
    #print(confusion_matrix(expect, predict))
    ConfusionMatrixDisplay(metrics.confusion_matrix(expect, predict), 
                           display_labels=sorted(target_names),
                          ).plot()
    

# Save Submission file

In [ ]:
df_submission = df_predict.copy(deep=True)

In [ ]:
df_submission.to_csv(f'maintask_TMUNLPG3_{log_time}.csv', index=False)